In [1]:
import pandas as pd
import numpy as np
!pip install optuna
import optuna
import tensorflow as tf
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score

# Load data
df = pd.read_csv('/kaggle/input/datasets/drshirshendulayek/fos-data/cleaned_data.csv')

# Drop empty rows
df = df.dropna()

# Define features and target
X = df.drop(columns=['FoS']).astype(float)
y = df['FoS'].astype(float)

# 1. SPLIT FIRST (Keeping Test data completely isolated)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. SCALE SECOND (Fitting ONLY on the training data)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Bayesian Optimization Objectives (Using Validation, NOT Test Data) ---

def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10)
    }
    model = RandomForestRegressor(**params, random_state=42)
    # Using 5-fold cross-validation on TRAINING data only to evaluate hyperparameters
    score = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2', n_jobs=-1).mean()
    return score

def objective_gbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10)
    }
    model = GradientBoostingRegressor(**params, random_state=42)
    # Using 5-fold cross-validation on TRAINING data only
    score = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2', n_jobs=-1).mean()
    return score

def objective_dl(trial):
    # For DL, CV is slow, so we do a quick train/val split purely from the training set
    X_t, X_v, y_t, y_v = train_test_split(X_train_scaled, y_train, test_size=0.2, random_state=42)

    n_layers = trial.suggest_int('n_layers', 1, 3)
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Dense(trial.suggest_int('units_0', 16, 128), activation='relu', input_shape=(X.shape[1],)))
    for i in range(1, n_layers):
        model.add(tf.keras.layers.Dense(trial.suggest_int(f'units_{i}', 16, 128), activation='relu'))
    model.add(tf.keras.layers.Dense(1))

    model.compile(optimizer='adam', loss='mse')
    # Train only on the inner training split, evaluate on the inner validation split
    model.fit(X_t, y_t, epochs=50, verbose=0, validation_data=(X_v, y_v))

    preds = model.predict(X_v)
    return r2_score(y_v, preds)

# --- Execute Optimization ---
print("Optimizing Random Forest...")
study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(objective_rf, n_trials=20)

print("Optimizing Gradient Boosting...")
study_gbm = optuna.create_study(direction='maximize')
study_gbm.optimize(objective_gbm, n_trials=20)

print("Optimizing Deep Learning...")
study_dl = optuna.create_study(direction='maximize')
study_dl.optimize(objective_dl, n_trials=10)

# --- Final Evaluation on the UNSEEN Test Set ---
print("\n--- Final Test Set Evaluation ---")

# Train best RF on full training data and test
best_rf = RandomForestRegressor(**study_rf.best_params, random_state=42)
best_rf.fit(X_train_scaled, y_train)
rf_test_score = r2_score(y_test, best_rf.predict(X_test_scaled))
print(f"Random Forest Final R2: {rf_test_score:.4f}")

# Train best GBM on full training data and test
best_gbm = GradientBoostingRegressor(**study_gbm.best_params, random_state=42)
best_gbm.fit(X_train_scaled, y_train)
gbm_test_score = r2_score(y_test, best_gbm.predict(X_test_scaled))
print(f"Gradient Boosting Final R2: {gbm_test_score:.4f}")

# Train best DL on full training data and test
best_dl = tf.keras.Sequential()
best_dl.add(tf.keras.layers.Dense(study_dl.best_params['units_0'], activation='relu', input_shape=(X.shape[1],)))
for i in range(1, study_dl.best_params['n_layers']):
    best_dl.add(tf.keras.layers.Dense(study_dl.best_params[f'units_{i}'], activation='relu'))
best_dl.add(tf.keras.layers.Dense(1))
best_dl.compile(optimizer='adam', loss='mse')
best_dl.fit(X_train_scaled, y_train, epochs=50, verbose=0)

dl_test_score = r2_score(y_test, best_dl.predict(X_test_scaled))
print(f"Deep Learning Final R2: {dl_test_score:.4f}")

[I 2026-07-09 09:52:45,520] A new study created in memory with name: no-name-869fd9f1-a4de-4d0e-9d34-0a4ee85a927b


Optimizing Random Forest...


[I 2026-07-09 09:53:05,519] Trial 0 finished with value: 0.9690878575685058 and parameters: {'n_estimators': 187, 'max_depth': 12, 'min_samples_split': 7}. Best is trial 0 with value: 0.9690878575685058.
[I 2026-07-09 09:53:29,931] Trial 1 finished with value: 0.9706054283482745 and parameters: {'n_estimators': 216, 'max_depth': 19, 'min_samples_split': 2}. Best is trial 1 with value: 0.9706054283482745.
[I 2026-07-09 09:53:52,411] Trial 2 finished with value: 0.9685974132910149 and parameters: {'n_estimators': 248, 'max_depth': 11, 'min_samples_split': 7}. Best is trial 1 with value: 0.9706054283482745.
[I 2026-07-09 09:54:03,059] Trial 3 finished with value: 0.9612737861164187 and parameters: {'n_estimators': 145, 'max_depth': 8, 'min_samples_split': 8}. Best is trial 1 with value: 0.9706054283482745.
[I 2026-07-09 09:54:20,626] Trial 4 finished with value: 0.9682032439281286 and parameters: {'n_estimators': 195, 'max_depth': 10, 'min_samples_split': 4}. Best is trial 1 with value: 0

Optimizing Gradient Boosting...


[I 2026-07-09 10:02:25,172] Trial 0 finished with value: 0.9715817608127001 and parameters: {'n_estimators': 147, 'learning_rate': 0.06588821984200384, 'max_depth': 9}. Best is trial 0 with value: 0.9715817608127001.
[I 2026-07-09 10:02:37,251] Trial 1 finished with value: 0.9721936483981246 and parameters: {'n_estimators': 133, 'learning_rate': 0.1938914030496778, 'max_depth': 6}. Best is trial 1 with value: 0.9721936483981246.
[I 2026-07-09 10:03:07,497] Trial 2 finished with value: 0.9722228802354081 and parameters: {'n_estimators': 336, 'learning_rate': 0.19463454631352087, 'max_depth': 6}. Best is trial 2 with value: 0.9722228802354081.
[I 2026-07-09 10:03:14,352] Trial 3 finished with value: 0.8519008125767187 and parameters: {'n_estimators': 68, 'learning_rate': 0.016240046653788413, 'max_depth': 7}. Best is trial 2 with value: 0.9722228802354081.
[I 2026-07-09 10:03:43,041] Trial 4 finished with value: 0.9703731363882779 and parameters: {'n_estimators': 464, 'learning_rate': 0.

...

73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


[I 2026-07-09 10:16:13,785] Trial 9 finished with value: 0.9538625810627536 and parameters: {'n_layers': 1, 'units_0': 78}. Best is trial 5 with value: 0.9717038845564929.



--- Final Test Set Evaluation ---
Random Forest Final R2: 0.9730
Gradient Boosting Final R2: 0.9758


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Deep Learning Final R2: 0.9578


In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor

# 1. Load Data
df = pd.read_csv('/kaggle/input/datasets/drshirshendulayek/fos-data/cleaned_data.csv')

# --- FIX: Standardize column names immediately ---
# This ensures consistency for the rest of the script
df.columns = ['Height', 'Slope_Angle', 'Density', 'Cohesion', 'Friction_Angle', 'FoS']

# 2. Prepare X and y
X = df.drop(columns=['FoS'])
y = df['FoS']

# 3. Train Surrogate Model (The "Physics Emulator")
# This model now learns patterns based on the simplified column names
surrogate = HistGradientBoostingRegressor(random_state=42)
surrogate.fit(X, y)

# 4. Generate new input grid
# We use the same simple names ('Height', 'Friction_Angle', etc.)
n_samples = 5000
new_inputs = pd.DataFrame({
    'Height': np.random.uniform(X['Height'].min(), X['Height'].max(), n_samples),
    'Slope_Angle': np.random.uniform(X['Slope_Angle'].min(), X['Slope_Angle'].max(), n_samples),
    'Density': np.random.uniform(X['Density'].min(), X['Density'].max(), n_samples),
    'Cohesion': np.random.uniform(X['Cohesion'].min(), X['Cohesion'].max(), n_samples),
    'Friction_Angle': np.random.uniform(X['Friction_Angle'].min(), X['Friction_Angle'].max(), n_samples)
})

# 5. Predict FEA-like results
new_inputs['FoS'] = surrogate.predict(new_inputs)

# 6. Save synthetic surrogate data
new_inputs.to_csv('/kaggle/working/fea_surrogate_data5k.csv', index=False)

print(f"Success! Synthetic data saved to 'fea_surrogate_data.csv'.")
print(f"Data shape: {new_inputs.shape}")

Success! Synthetic data saved to 'fea_surrogate_data.csv'.
Data shape: (5000, 6)


In [3]:
import pandas as pd

# 1. Load the files
df_original = pd.read_csv('/kaggle/input/datasets/drshirshendulayek/fos-data/cleaned_data.csv')
df_surrogate = pd.read_csv('/kaggle/working/fea_surrogate_data5k.csv')

# 2. Standardize columns to ensure perfect alignment
# We use the clean names from our previous steps
cols = ['Height', 'Slope_Angle', 'Density', 'Cohesion', 'Friction_Angle', 'FoS']
df_original.columns = cols
df_surrogate.columns = cols

# 3. Concatenate
df_combined = pd.concat([df_original, df_surrogate], axis=0, ignore_index=True)

# 4. Shuffle (Crucial for preventing bias)
# This ensures synthetic data is distributed evenly during training
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

# 5. Export for final training
df_combined.to_csv('/kaggle/working/merged_data5k.csv', index=False)

print(f"Merge successful!")
print(f"Original count: {len(df_original)}")
print(f"Surrogate count: {len(df_surrogate)}")
print(f"Total training data size: {len(df_combined)}")

Merge successful!
Original count: 14518
Surrogate count: 5000
Total training data size: 19518


In [4]:
import pandas as pd
import numpy as np
!pip install optuna
import optuna
import tensorflow as tf
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score

# Load data
df = pd.read_csv('/kaggle/working/merged_data5k.csv')

# Drop empty rows
df = df.dropna()

# Define features and target
X = df.drop(columns=['FoS']).astype(float)
y = df['FoS'].astype(float)

# 1. SPLIT FIRST (Keeping Test data completely isolated)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. SCALE SECOND (Fitting ONLY on the training data)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Bayesian Optimization Objectives (Using Validation, NOT Test Data) ---

def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10)
    }
    model = RandomForestRegressor(**params, random_state=42)
    # Using 5-fold cross-validation on TRAINING data only to evaluate hyperparameters
    score = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2', n_jobs=-1).mean()
    return score

def objective_gbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10)
    }
    model = GradientBoostingRegressor(**params, random_state=42)
    # Using 5-fold cross-validation on TRAINING data only
    score = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2', n_jobs=-1).mean()
    return score

def objective_dl(trial):
    # For DL, CV is slow, so we do a quick train/val split purely from the training set
    X_t, X_v, y_t, y_v = train_test_split(X_train_scaled, y_train, test_size=0.2, random_state=42)

    n_layers = trial.suggest_int('n_layers', 1, 3)
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Dense(trial.suggest_int('units_0', 16, 128), activation='relu', input_shape=(X.shape[1],)))
    for i in range(1, n_layers):
        model.add(tf.keras.layers.Dense(trial.suggest_int(f'units_{i}', 16, 128), activation='relu'))
    model.add(tf.keras.layers.Dense(1))

    model.compile(optimizer='adam', loss='mse')
    # Train only on the inner training split, evaluate on the inner validation split
    model.fit(X_t, y_t, epochs=50, verbose=0, validation_data=(X_v, y_v))

    preds = model.predict(X_v)
    return r2_score(y_v, preds)

# --- Execute Optimization ---
print("Optimizing Random Forest...")
study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(objective_rf, n_trials=20)

print("Optimizing Gradient Boosting...")
study_gbm = optuna.create_study(direction='maximize')
study_gbm.optimize(objective_gbm, n_trials=20)

print("Optimizing Deep Learning...")
study_dl = optuna.create_study(direction='maximize')
study_dl.optimize(objective_dl, n_trials=10)

# --- Final Evaluation on the UNSEEN Test Set ---
print("\n--- Final Test Set Evaluation ---")

# Train best RF on full training data and test
best_rf = RandomForestRegressor(**study_rf.best_params, random_state=42)
best_rf.fit(X_train_scaled, y_train)
rf_test_score = r2_score(y_test, best_rf.predict(X_test_scaled))
print(f"Random Forest Final R2: {rf_test_score:.4f}")

# Train best GBM on full training data and test
best_gbm = GradientBoostingRegressor(**study_gbm.best_params, random_state=42)
best_gbm.fit(X_train_scaled, y_train)
gbm_test_score = r2_score(y_test, best_gbm.predict(X_test_scaled))
print(f"Gradient Boosting Final R2: {gbm_test_score:.4f}")

# Train best DL on full training data and test
best_dl = tf.keras.Sequential()
best_dl.add(tf.keras.layers.Dense(study_dl.best_params['units_0'], activation='relu', input_shape=(X.shape[1],)))
for i in range(1, study_dl.best_params['n_layers']):
    best_dl.add(tf.keras.layers.Dense(study_dl.best_params[f'units_{i}'], activation='relu'))
best_dl.add(tf.keras.layers.Dense(1))
best_dl.compile(optimizer='adam', loss='mse')
best_dl.fit(X_train_scaled, y_train, epochs=50, verbose=0)

dl_test_score = r2_score(y_test, best_dl.predict(X_test_scaled))
print(f"Deep Learning Final R2: {dl_test_score:.4f}")

[I 2026-07-09 10:17:42,902] A new study created in memory with name: no-name-bee69125-fdd7-46e9-8cbc-72969434317d


Optimizing Random Forest...


[I 2026-07-09 10:18:11,415] Trial 0 finished with value: 0.9752599510863551 and parameters: {'n_estimators': 215, 'max_depth': 10, 'min_samples_split': 8}. Best is trial 0 with value: 0.9752599510863551.
[I 2026-07-09 10:19:23,041] Trial 1 finished with value: 0.9784690818013468 and parameters: {'n_estimators': 458, 'max_depth': 16, 'min_samples_split': 5}. Best is trial 1 with value: 0.9784690818013468.
[I 2026-07-09 10:19:58,753] Trial 2 finished with value: 0.9778107179376491 and parameters: {'n_estimators': 236, 'max_depth': 19, 'min_samples_split': 8}. Best is trial 1 with value: 0.9784690818013468.
[I 2026-07-09 10:20:24,092] Trial 3 finished with value: 0.9784941610419613 and parameters: {'n_estimators': 164, 'max_depth': 15, 'min_samples_split': 4}. Best is trial 3 with value: 0.9784941610419613.
[I 2026-07-09 10:20:26,538] Trial 4 finished with value: 0.8384756714676997 and parameters: {'n_estimators': 57, 'max_depth': 3, 'min_samples_split': 7}. Best is trial 3 with value: 0.

Optimizing Gradient Boosting...


[I 2026-07-09 10:32:37,863] Trial 0 finished with value: 0.9687650984105718 and parameters: {'n_estimators': 322, 'learning_rate': 0.08474335543207749, 'max_depth': 3}. Best is trial 0 with value: 0.9687650984105718.
[I 2026-07-09 10:33:34,216] Trial 1 finished with value: 0.9780629230658194 and parameters: {'n_estimators': 435, 'learning_rate': 0.219049575182995, 'max_depth': 6}. Best is trial 1 with value: 0.9780629230658194.
[I 2026-07-09 10:34:21,526] Trial 2 finished with value: 0.9778050586161608 and parameters: {'n_estimators': 248, 'learning_rate': 0.21277422447826191, 'max_depth': 9}. Best is trial 1 with value: 0.9780629230658194.
[I 2026-07-09 10:35:20,348] Trial 3 finished with value: 0.9779859388192989 and parameters: {'n_estimators': 347, 'learning_rate': 0.1570394700961227, 'max_depth': 8}. Best is trial 1 with value: 0.9780629230658194.
[I 2026-07-09 10:36:30,217] Trial 4 finished with value: 0.9769344130370007 and parameters: {'n_estimators': 331, 'learning_rate': 0.17

...

98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


[I 2026-07-09 10:53:51,549] Trial 9 finished with value: 0.9668189564409434 and parameters: {'n_layers': 2, 'units_0': 53, 'units_1': 98}. Best is trial 5 with value: 0.9669026128391031.



--- Final Test Set Evaluation ---
Random Forest Final R2: 0.9797
Gradient Boosting Final R2: 0.9820


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


122/122 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Deep Learning Final R2: 0.9683


In [5]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor

# 1. Load Data
df = pd.read_csv('/kaggle/input/datasets/drshirshendulayek/fos-data/cleaned_data.csv')

# --- FIX: Standardize column names immediately ---
# This ensures consistency for the rest of the script
df.columns = ['Height', 'Slope_Angle', 'Density', 'Cohesion', 'Friction_Angle', 'FoS']

# 2. Prepare X and y
X = df.drop(columns=['FoS'])
y = df['FoS']

# 3. Train Surrogate Model (The "Physics Emulator")
# This model now learns patterns based on the simplified column names
surrogate = HistGradientBoostingRegressor(random_state=42)
surrogate.fit(X, y)

# 4. Generate new input grid
# We use the same simple names ('Height', 'Friction_Angle', etc.)
n_samples = 10000
new_inputs = pd.DataFrame({
    'Height': np.random.uniform(X['Height'].min(), X['Height'].max(), n_samples),
    'Slope_Angle': np.random.uniform(X['Slope_Angle'].min(), X['Slope_Angle'].max(), n_samples),
    'Density': np.random.uniform(X['Density'].min(), X['Density'].max(), n_samples),
    'Cohesion': np.random.uniform(X['Cohesion'].min(), X['Cohesion'].max(), n_samples),
    'Friction_Angle': np.random.uniform(X['Friction_Angle'].min(), X['Friction_Angle'].max(), n_samples)
})

# 5. Predict FEA-like results
new_inputs['FoS'] = surrogate.predict(new_inputs)

# 6. Save synthetic surrogate data
new_inputs.to_csv('/kaggle/working/fea_surrogate_data10.csv', index=False)

print(f"Success! Synthetic data saved to 'fea_surrogate_data.csv'.")
print(f"Data shape: {new_inputs.shape}")

Success! Synthetic data saved to 'fea_surrogate_data.csv'.
Data shape: (10000, 6)


In [6]:
import pandas as pd

# 1. Load the files
df_original = pd.read_csv('/kaggle/input/datasets/drshirshendulayek/fos-data/cleaned_data.csv')
df_surrogate = pd.read_csv('/kaggle/working/fea_surrogate_data10.csv')

# 2. Standardize columns to ensure perfect alignment
# We use the clean names from our previous steps
cols = ['Height', 'Slope_Angle', 'Density', 'Cohesion', 'Friction_Angle', 'FoS']
df_original.columns = cols
df_surrogate.columns = cols

# 3. Concatenate
df_combined = pd.concat([df_original, df_surrogate], axis=0, ignore_index=True)

# 4. Shuffle (Crucial for preventing bias)
# This ensures synthetic data is distributed evenly during training
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

# 5. Export for final training
df_combined.to_csv('merged_data10k.csv', index=False)

print(f"Merge successful!")
print(f"Original count: {len(df_original)}")
print(f"Surrogate count: {len(df_surrogate)}")
print(f"Total training data size: {len(df_combined)}")

Merge successful!
Original count: 14518
Surrogate count: 10000
Total training data size: 24518


In [7]:
import pandas as pd
import numpy as np
!pip install optuna
import optuna
import tensorflow as tf
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score

# Load data
df = pd.read_csv('/kaggle/working/merged_data10k.csv')

# Drop empty rows
df = df.dropna()

# Define features and target
X = df.drop(columns=['FoS']).astype(float)
y = df['FoS'].astype(float)

# 1. SPLIT FIRST (Keeping Test data completely isolated)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. SCALE SECOND (Fitting ONLY on the training data)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Bayesian Optimization Objectives (Using Validation, NOT Test Data) ---

def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10)
    }
    model = RandomForestRegressor(**params, random_state=42)
    # Using 5-fold cross-validation on TRAINING data only to evaluate hyperparameters
    score = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2', n_jobs=-1).mean()
    return score

def objective_gbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10)
    }
    model = GradientBoostingRegressor(**params, random_state=42)
    # Using 5-fold cross-validation on TRAINING data only
    score = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2', n_jobs=-1).mean()
    return score

def objective_dl(trial):
    # For DL, CV is slow, so we do a quick train/val split purely from the training set
    X_t, X_v, y_t, y_v = train_test_split(X_train_scaled, y_train, test_size=0.2, random_state=42)

    n_layers = trial.suggest_int('n_layers', 1, 3)
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Dense(trial.suggest_int('units_0', 16, 128), activation='relu', input_shape=(X.shape[1],)))
    for i in range(1, n_layers):
        model.add(tf.keras.layers.Dense(trial.suggest_int(f'units_{i}', 16, 128), activation='relu'))
    model.add(tf.keras.layers.Dense(1))

    model.compile(optimizer='adam', loss='mse')
    # Train only on the inner training split, evaluate on the inner validation split
    model.fit(X_t, y_t, epochs=50, verbose=0, validation_data=(X_v, y_v))

    preds = model.predict(X_v)
    return r2_score(y_v, preds)

# --- Execute Optimization ---
print("Optimizing Random Forest...")
study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(objective_rf, n_trials=20)

print("Optimizing Gradient Boosting...")
study_gbm = optuna.create_study(direction='maximize')
study_gbm.optimize(objective_gbm, n_trials=20)

print("Optimizing Deep Learning...")
study_dl = optuna.create_study(direction='maximize')
study_dl.optimize(objective_dl, n_trials=10)

# --- Final Evaluation on the UNSEEN Test Set ---
print("\n--- Final Test Set Evaluation ---")

# Train best RF on full training data and test
best_rf = RandomForestRegressor(**study_rf.best_params, random_state=42)
best_rf.fit(X_train_scaled, y_train)
rf_test_score = r2_score(y_test, best_rf.predict(X_test_scaled))
print(f"Random Forest Final R2: {rf_test_score:.4f}")

# Train best GBM on full training data and test
best_gbm = GradientBoostingRegressor(**study_gbm.best_params, random_state=42)
best_gbm.fit(X_train_scaled, y_train)
gbm_test_score = r2_score(y_test, best_gbm.predict(X_test_scaled))
print(f"Gradient Boosting Final R2: {gbm_test_score:.4f}")

# Train best DL on full training data and test
best_dl = tf.keras.Sequential()
best_dl.add(tf.keras.layers.Dense(study_dl.best_params['units_0'], activation='relu', input_shape=(X.shape[1],)))
for i in range(1, study_dl.best_params['n_layers']):
    best_dl.add(tf.keras.layers.Dense(study_dl.best_params[f'units_{i}'], activation='relu'))
best_dl.add(tf.keras.layers.Dense(1))
best_dl.compile(optimizer='adam', loss='mse')
best_dl.fit(X_train_scaled, y_train, epochs=50, verbose=0)

dl_test_score = r2_score(y_test, best_dl.predict(X_test_scaled))
print(f"Deep Learning Final R2: {dl_test_score:.4f}")

[I 2026-07-09 10:55:35,752] A new study created in memory with name: no-name-880a7e72-ed8b-45d4-b9ec-73b6cfad81df


Optimizing Random Forest...


[I 2026-07-09 10:55:55,509] Trial 0 finished with value: 0.9306518348988808 and parameters: {'n_estimators': 205, 'max_depth': 5, 'min_samples_split': 3}. Best is trial 0 with value: 0.9306518348988808.
[I 2026-07-09 10:56:03,370] Trial 1 finished with value: 0.8319421831398188 and parameters: {'n_estimators': 144, 'max_depth': 3, 'min_samples_split': 9}. Best is trial 0 with value: 0.9306518348988808.
[I 2026-07-09 10:56:28,545] Trial 2 finished with value: 0.8966487495245861 and parameters: {'n_estimators': 358, 'max_depth': 4, 'min_samples_split': 9}. Best is trial 0 with value: 0.9306518348988808.
[I 2026-07-09 10:57:34,881] Trial 3 finished with value: 0.9769650041478641 and parameters: {'n_estimators': 421, 'max_depth': 10, 'min_samples_split': 5}. Best is trial 3 with value: 0.9769650041478641.
[I 2026-07-09 10:58:31,925] Trial 4 finished with value: 0.9794727450942593 and parameters: {'n_estimators': 269, 'max_depth': 17, 'min_samples_split': 3}. Best is trial 4 with value: 0.9

Optimizing Gradient Boosting...


[I 2026-07-09 11:09:11,166] Trial 0 finished with value: 0.9785514418585768 and parameters: {'n_estimators': 383, 'learning_rate': 0.1906917067525817, 'max_depth': 4}. Best is trial 0 with value: 0.9785514418585768.
[I 2026-07-09 11:09:36,291] Trial 1 finished with value: 0.979277523597017 and parameters: {'n_estimators': 176, 'learning_rate': 0.11173992870978998, 'max_depth': 5}. Best is trial 1 with value: 0.979277523597017.
[I 2026-07-09 11:11:20,662] Trial 2 finished with value: 0.9801572356280062 and parameters: {'n_estimators': 435, 'learning_rate': 0.011659140190525063, 'max_depth': 9}. Best is trial 2 with value: 0.9801572356280062.
[I 2026-07-09 11:12:48,428] Trial 3 finished with value: 0.9810949083048996 and parameters: {'n_estimators': 378, 'learning_rate': 0.13314827172804639, 'max_depth': 8}. Best is trial 3 with value: 0.9810949083048996.
[I 2026-07-09 11:13:10,234] Trial 4 finished with value: 0.9512534552193019 and parameters: {'n_estimators': 84, 'learning_rate': 0.02

...

123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


[I 2026-07-09 11:37:46,039] Trial 9 finished with value: 0.9673074422976833 and parameters: {'n_layers': 2, 'units_0': 43, 'units_1': 93}. Best is trial 1 with value: 0.9701799147025676.



--- Final Test Set Evaluation ---
Random Forest Final R2: 0.9853
Gradient Boosting Final R2: 0.9871


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Deep Learning Final R2: 0.9698


In [8]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor

# 1. Load Data
df = pd.read_csv('/kaggle/input/datasets/drshirshendulayek/fos-data/cleaned_data.csv')

# --- FIX: Standardize column names immediately ---
# This ensures consistency for the rest of the script
df.columns = ['Height', 'Slope_Angle', 'Density', 'Cohesion', 'Friction_Angle', 'FoS']

# 2. Prepare X and y
X = df.drop(columns=['FoS'])
y = df['FoS']

# 3. Train Surrogate Model (The "Physics Emulator")
# This model now learns patterns based on the simplified column names
surrogate = HistGradientBoostingRegressor(random_state=42)
surrogate.fit(X, y)

# 4. Generate new input grid
# We use the same simple names ('Height', 'Friction_Angle', etc.)
n_samples = 15000
new_inputs = pd.DataFrame({
    'Height': np.random.uniform(X['Height'].min(), X['Height'].max(), n_samples),
    'Slope_Angle': np.random.uniform(X['Slope_Angle'].min(), X['Slope_Angle'].max(), n_samples),
    'Density': np.random.uniform(X['Density'].min(), X['Density'].max(), n_samples),
    'Cohesion': np.random.uniform(X['Cohesion'].min(), X['Cohesion'].max(), n_samples),
    'Friction_Angle': np.random.uniform(X['Friction_Angle'].min(), X['Friction_Angle'].max(), n_samples)
})

# 5. Predict FEA-like results
new_inputs['FoS'] = surrogate.predict(new_inputs)

# 6. Save synthetic surrogate data
new_inputs.to_csv('/kaggle/working/fea_surrogate_data15k.csv', index=False)

print(f"Success! Synthetic data saved to 'fea_surrogate_data.csv'.")
print(f"Data shape: {new_inputs.shape}")

Success! Synthetic data saved to 'fea_surrogate_data.csv'.
Data shape: (15000, 6)


In [9]:
import pandas as pd

# 1. Load the files
df_original = pd.read_csv('/kaggle/input/datasets/drshirshendulayek/fos-data/cleaned_data.csv')
df_surrogate = pd.read_csv('/kaggle/working/fea_surrogate_data15k.csv')

# 2. Standardize columns to ensure perfect alignment
# We use the clean names from our previous steps
cols = ['Height', 'Slope_Angle', 'Density', 'Cohesion', 'Friction_Angle', 'FoS']
df_original.columns = cols
df_surrogate.columns = cols

# 3. Concatenate
df_combined = pd.concat([df_original, df_surrogate], axis=0, ignore_index=True)

# 4. Shuffle (Crucial for preventing bias)
# This ensures synthetic data is distributed evenly during training
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

# 5. Export for final training
df_combined.to_csv('/kaggle/working/merged_data15k.csv', index=False)

print(f"Merge successful!")
print(f"Original count: {len(df_original)}")
print(f"Surrogate count: {len(df_surrogate)}")
print(f"Total training data size: {len(df_combined)}")

Merge successful!
Original count: 14518
Surrogate count: 15000
Total training data size: 29518


In [10]:
import pandas as pd
import numpy as np
!pip install optuna
import optuna
import tensorflow as tf
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score

# Load data
df = pd.read_csv('/kaggle/working/merged_data15k.csv')

# Drop empty rows
df = df.dropna()

# Define features and target
X = df.drop(columns=['FoS']).astype(float)
y = df['FoS'].astype(float)

# 1. SPLIT FIRST (Keeping Test data completely isolated)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. SCALE SECOND (Fitting ONLY on the training data)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Bayesian Optimization Objectives (Using Validation, NOT Test Data) ---

def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10)
    }
    model = RandomForestRegressor(**params, random_state=42)
    # Using 5-fold cross-validation on TRAINING data only to evaluate hyperparameters
    score = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2', n_jobs=-1).mean()
    return score

def objective_gbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10)
    }
    model = GradientBoostingRegressor(**params, random_state=42)
    # Using 5-fold cross-validation on TRAINING data only
    score = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2', n_jobs=-1).mean()
    return score

def objective_dl(trial):
    # For DL, CV is slow, so we do a quick train/val split purely from the training set
    X_t, X_v, y_t, y_v = train_test_split(X_train_scaled, y_train, test_size=0.2, random_state=42)

    n_layers = trial.suggest_int('n_layers', 1, 3)
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Dense(trial.suggest_int('units_0', 16, 128), activation='relu', input_shape=(X.shape[1],)))
    for i in range(1, n_layers):
        model.add(tf.keras.layers.Dense(trial.suggest_int(f'units_{i}', 16, 128), activation='relu'))
    model.add(tf.keras.layers.Dense(1))

    model.compile(optimizer='adam', loss='mse')
    # Train only on the inner training split, evaluate on the inner validation split
    model.fit(X_t, y_t, epochs=50, verbose=0, validation_data=(X_v, y_v))

    preds = model.predict(X_v)
    return r2_score(y_v, preds)

# --- Execute Optimization ---
print("Optimizing Random Forest...")
study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(objective_rf, n_trials=20)

print("Optimizing Gradient Boosting...")
study_gbm = optuna.create_study(direction='maximize')
study_gbm.optimize(objective_gbm, n_trials=20)

print("Optimizing Deep Learning...")
study_dl = optuna.create_study(direction='maximize')
study_dl.optimize(objective_dl, n_trials=10)

# --- Final Evaluation on the UNSEEN Test Set ---
print("\n--- Final Test Set Evaluation ---")

# Train best RF on full training data and test
best_rf = RandomForestRegressor(**study_rf.best_params, random_state=42)
best_rf.fit(X_train_scaled, y_train)
rf_test_score = r2_score(y_test, best_rf.predict(X_test_scaled))
print(f"Random Forest Final R2: {rf_test_score:.4f}")

# Train best GBM on full training data and test
best_gbm = GradientBoostingRegressor(**study_gbm.best_params, random_state=42)
best_gbm.fit(X_train_scaled, y_train)
gbm_test_score = r2_score(y_test, best_gbm.predict(X_test_scaled))
print(f"Gradient Boosting Final R2: {gbm_test_score:.4f}")

# Train best DL on full training data and test
best_dl = tf.keras.Sequential()
best_dl.add(tf.keras.layers.Dense(study_dl.best_params['units_0'], activation='relu', input_shape=(X.shape[1],)))
for i in range(1, study_dl.best_params['n_layers']):
    best_dl.add(tf.keras.layers.Dense(study_dl.best_params[f'units_{i}'], activation='relu'))
best_dl.add(tf.keras.layers.Dense(1))
best_dl.compile(optimizer='adam', loss='mse')
best_dl.fit(X_train_scaled, y_train, epochs=50, verbose=0)

dl_test_score = r2_score(y_test, best_dl.predict(X_test_scaled))
print(f"Deep Learning Final R2: {dl_test_score:.4f}")

[I 2026-07-09 11:39:42,595] A new study created in memory with name: no-name-9dda6a54-a33c-4936-a702-eaf5602f12e5


Optimizing Random Forest...


[I 2026-07-09 11:40:45,151] Trial 0 finished with value: 0.9834918136569357 and parameters: {'n_estimators': 263, 'max_depth': 12, 'min_samples_split': 3}. Best is trial 0 with value: 0.9834918136569357.
[I 2026-07-09 11:41:15,416] Trial 1 finished with value: 0.9836699907999605 and parameters: {'n_estimators': 112, 'max_depth': 19, 'min_samples_split': 5}. Best is trial 1 with value: 0.9836699907999605.
[I 2026-07-09 11:42:51,788] Trial 2 finished with value: 0.9829803313657675 and parameters: {'n_estimators': 384, 'max_depth': 17, 'min_samples_split': 10}. Best is trial 1 with value: 0.9836699907999605.
[I 2026-07-09 11:43:07,998] Trial 3 finished with value: 0.9514146177819252 and parameters: {'n_estimators': 129, 'max_depth': 6, 'min_samples_split': 5}. Best is trial 1 with value: 0.9836699907999605.
[I 2026-07-09 11:44:18,780] Trial 4 finished with value: 0.9783242252058351 and parameters: {'n_estimators': 385, 'max_depth': 9, 'min_samples_split': 2}. Best is trial 1 with value: 0

Optimizing Gradient Boosting...


[I 2026-07-09 12:05:21,974] Trial 0 finished with value: 0.9600222215148371 and parameters: {'n_estimators': 280, 'learning_rate': 0.031884749049286364, 'max_depth': 3}. Best is trial 0 with value: 0.9600222215148371.
[I 2026-07-09 12:05:55,600] Trial 1 finished with value: 0.9695694077918177 and parameters: {'n_estimators': 233, 'learning_rate': 0.028174773119110437, 'max_depth': 4}. Best is trial 1 with value: 0.9695694077918177.
[I 2026-07-09 12:06:57,481] Trial 2 finished with value: 0.9845742293379193 and parameters: {'n_estimators': 245, 'learning_rate': 0.2308040147851346, 'max_depth': 7}. Best is trial 2 with value: 0.9845742293379193.
[I 2026-07-09 12:07:23,721] Trial 3 finished with value: 0.9804947807930043 and parameters: {'n_estimators': 181, 'learning_rate': 0.191360968496285, 'max_depth': 4}. Best is trial 2 with value: 0.9845742293379193.
[I 2026-07-09 12:08:08,747] Trial 4 finished with value: 0.9614252275372012 and parameters: {'n_estimators': 409, 'learning_rate': 0.

...

148/148 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


[I 2026-07-09 12:32:50,967] Trial 9 finished with value: 0.9700677542845507 and parameters: {'n_layers': 2, 'units_0': 125, 'units_1': 34}. Best is trial 6 with value: 0.9734306086762834.



--- Final Test Set Evaluation ---
Random Forest Final R2: 0.9808
Gradient Boosting Final R2: 0.9819


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


185/185 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Deep Learning Final R2: 0.9665
